In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os
import re
import pytz
from datetime import datetime

In [2]:
apps_df = pd.read_csv("data/Play Store Data.csv")
reviews_df = pd.read_csv("data/User Reviews.csv")

In [3]:
apps_df.shape

(10841, 13)

In [4]:
reviews_df.shape

(64295, 5)

In [5]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [6]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [7]:
apps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [8]:
apps_df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [9]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  object 
 1   Translated_Review       37427 non-null  object 
 2   Sentiment               37432 non-null  object 
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), object(3)
memory usage: 2.5+ MB


In [10]:
reviews_df.isnull().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

In [11]:
apps_df.duplicated().sum()
apps_df.duplicated(subset="App").sum()

np.int64(1181)

In [12]:
apps_df = apps_df.drop_duplicates(subset="App")

In [13]:
def convert_size(size):
    size = str(size)

    if size.endswith("M"):
        return float(size[:-1])

    if size.endswith("k"):
        return float(size[:-1])/1024

    return None

In [14]:
apps_df["Size_MB"] = apps_df["Size"].apply(convert_size)
print(apps_df["Size_MB"].isnull().sum())

1228


In [15]:
apps_df["Rating"] = pd.to_numeric(apps_df["Rating"],errors="coerce")
apps_df = apps_df.dropna(subset=["Rating"])
print(apps_df["Rating"].isnull().sum())

0


In [16]:
apps_df["Installs"] = (apps_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

apps_df["Installs"] = pd.to_numeric(
    apps_df["Installs"],
    errors="coerce"
)

apps_df["Installs"].isnull().sum()

np.int64(1)

In [17]:
apps_df.loc[apps_df["Installs"].isnull()]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",NaN,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN,NaN


In [18]:
apps_df = apps_df[apps_df["Installs"].notnull()]

In [19]:
apps_df["Reviews"] = pd.to_numeric(apps_df["Reviews"],errors="coerce")
print(apps_df["Reviews"].isnull().sum())


0


In [20]:
apps_df["Category"] = (apps_df["Category"].astype(str).str.upper())

In [21]:
reviews_df["Sentiment_Subjectivity"] = pd.to_numeric(reviews_df["Sentiment_Subjectivity"],errors="coerce")
reviews_df = reviews_df.dropna(
    subset=["Sentiment_Subjectivity"]
)

reviews_df["Sentiment_Subjectivity"].isnull().sum()

np.int64(0)

In [22]:
reviews_df = reviews_df.drop_duplicates()
reviews_df.shape

(29697, 5)

In [23]:
sentiment_avg = (reviews_df.groupby("App")["Sentiment_Subjectivity"].mean().reset_index())
sentiment_avg.head()

,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.493254
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.508907
2,11st,0.443957
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


In [24]:
merged_df = apps_df.merge(sentiment_avg, on="App",how="inner")
merged_df.shape

(816, 15)

In [25]:
apps_df["Last Updated"] = pd.to_datetime(apps_df["Last Updated"], errors="coerce")
apps_df = apps_df.dropna(subset=["Last Updated"])

def parse_android_ver(v):
    v = str(v)
    match = re.match(r"(\d+(\.\d+)?)", v)
    return float(match.group(1)) if match else None

apps_df["Android_Ver_Num"] = apps_df["Android Ver"].apply(parse_android_ver)

apps_df["Price_Clean"] = apps_df["Price"].astype(str).str.replace("$", "", regex=False)
apps_df["Price_Clean"] = pd.to_numeric(apps_df["Price_Clean"], errors="coerce").fillna(0)
apps_df["Revenue"] = apps_df["Price_Clean"] * apps_df["Installs"]

In [26]:
allowed_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

In [27]:
filtered_df = merged_df[
    (merged_df["Rating"] > 3.5) &
    (merged_df["Reviews"] > 500) &
    (merged_df["Installs"] > 50000) &
    (merged_df["Sentiment_Subjectivity"] > 0.5) &
    (merged_df["Category"].isin(allowed_categories)) &
    (~merged_df["App"].str.contains("s", case=False, na=False)) &
    (merged_df["Size_MB"].notnull())
].copy()

In [28]:
filtered_df.shape

(25, 15)

In [29]:
translation_map = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Verabredung"
}

filtered_df["Category_Display"] = filtered_df["Category"].replace(translation_map)


In [30]:
filtered_df["Color"] = np.where(
    filtered_df["Category"] == "GAME",
    "GAME",
    "OTHER"
)

In [31]:
import pytz
from datetime import datetime

india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour

if 17 <= current_hour < 19:

    fig = px.scatter(
        filtered_df,
        x="Size_MB",
        y="Rating",
        size="Installs",
        color="Color",
        color_discrete_map={"GAME": "pink", "OTHER": "steelblue"},
        hover_name="App",
        hover_data={"Category_Display": True, "Installs": True, "Reviews": True}, 
        title="Google Play Store Bubble Chart"
    )

    fig.show()

    os.makedirs("outputs", exist_ok=True)
    fig.write_html("outputs/task1_bubble_chart.html")

else:
    print("This chart is only available between 5 PM and 7 PM IST.")

This chart is only available between 5 PM and 7 PM IST.


In [32]:
category_installs = apps_df.groupby("Category")["Installs"].sum().reset_index()
category_installs.head()

,Category,Installs
0,ART_AND_DESIGN,1.142331e+08
1,AUTO_AND_VEHICLES,5.312980e+07
2,BEAUTY,2.691620e+07
3,BOOKS_AND_REFERENCE,1.665792e+09
4,BUSINESS,6.970181e+08


In [33]:
category_installs = category_installs[
    ~category_installs["Category"].str.startswith(("A", "C", "G", "S"))
]
category_installs.shape

(25, 2)

In [34]:
top5_categories = category_installs.sort_values("Installs", ascending=False).head(5)
top5_categories

,Category,Installs
29,TOOLS,7.999724e+09
25,PRODUCTIVITY,5.793070e+09
24,PHOTOGRAPHY,4.649143e+09
11,FAMILY,4.427480e+09
31,VIDEO_PLAYERS,3.926797e+09


In [35]:
top5_categories["Highlighted"] = top5_categories["Installs"] > 1_000_000

top5_categories["Category_Label"] = top5_categories.apply(
    lambda r: f"{r['Category']} ⭐ (>1M installs)" if r["Highlighted"] else r["Category"],
    axis=1
)
top5_categories

,Category,Installs,Highlighted,Category_Label
29,TOOLS,7.999724e+09,True,TOOLS ⭐ (>1M installs)
25,PRODUCTIVITY,5.793070e+09,True,PRODUCTIVITY ⭐ (>1M installs)
24,PHOTOGRAPHY,4.649143e+09,True,PHOTOGRAPHY ⭐ (>1M installs)
11,FAMILY,4.427480e+09,True,FAMILY ⭐ (>1M installs)
31,VIDEO_PLAYERS,3.926797e+09,True,VIDEO_PLAYERS ⭐ (>1M installs)


### Note on geographic distribution methodology
The Play Store dataset has no country-level install breakdown. To render a meaningful 
choropleth, installs are distributed across countries using population share as a proxy 
weight (using `px.data.gapminder()` population data) — i.e., countries with larger 
populations are assumed to account for a proportionally larger share of a category's 
global installs. This is a documented estimation technique, not raw data, and is called 
out here per mentor guidance.

In [36]:
gapminder = px.data.gapminder().drop_duplicates("iso_alpha")[["iso_alpha", "country", "pop"]]
world_pop = gapminder["pop"].sum()

geo_rows = []
for _, row in top5_categories.iterrows():
    for _, g in gapminder.iterrows():
        weighted_installs = row["Installs"] * (g["pop"] / world_pop)
        geo_rows.append({
            "iso_alpha": g["iso_alpha"],
            "country": g["country"],
            "Category_Label": row["Category_Label"],
            "Installs": weighted_installs,
            "Highlighted": row["Highlighted"]
        })

geo_df = pd.DataFrame(geo_rows)
geo_df.shape

(705, 5)

In [37]:
india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour

if 18 <= current_hour < 20:

    fig2 = px.choropleth(
        geo_df,
        locations="iso_alpha",
        color="Installs",
        animation_frame="Category_Label",
        hover_name="country",
        color_continuous_scale=px.colors.sequential.Plasma,
        title="Global Installs by Category (Top 5, excluding A/C/G/S) — Population-Weighted Estimate"
    )

    # Visual highlight: red border on frames where category exceeds 1M installs
    for frame in fig2.frames:
        label = frame.name
        is_highlighted = geo_df.loc[geo_df["Category_Label"] == label, "Highlighted"].iloc[0]
        if is_highlighted:
            frame.data[0].marker.line.width = 1.5
            frame.data[0].marker.line.color = "red"

    fig2.update_geos(
        projection_type="natural earth",
        showcountries=True,
        showcoastlines=True,
        scope="world"
    )

    fig2.update_layout(
        width=1000,
        height=600,
        margin=dict(l=0, r=0, t=60, b=0)
    )

    fig2.show()

    os.makedirs("outputs", exist_ok=True)
    fig2.write_html("outputs/task2_choropleth.html")

else:
    print("This chart is only available between 6 PM and 8 PM IST.")

This chart is only available between 6 PM and 8 PM IST.


**Note on >1M installs highlight:** The highlight condition (red border + ⭐) is applied 
correctly per the task requirement, but since the top 5 categories are selected specifically 
because they have the highest total installs in the dataset, all 5 categories exceed the 
1M install threshold by definition. As a result, every animation frame shows the highlighted 
state — this is expected behavior given the underlying data distribution, not a logic error.

**Insight:** Even as an estimate, the map consistently shows India and China with the 
highest projected installs across categories — consistent with these being among the 
largest Android user bases globally, so the estimation aligns with real-world expectations.


In [38]:
allowed_prefixes_task3 = ("E", "C", "B")

ts_filtered = apps_df[
    (apps_df["Reviews"] > 500) &
    (apps_df["Category"].str.startswith(allowed_prefixes_task3)) &
    (~apps_df["App"].str.lower().str.startswith(("x", "y", "z"))) &
    (~apps_df["App"].str.contains("s", case=False, na=False))
].copy()

ts_filtered.shape

(197, 17)

In [39]:
ts_filtered["Month"] = ts_filtered["Last Updated"].dt.to_period("M").dt.to_timestamp()

monthly_installs = (
    ts_filtered.groupby(["Category", "Month"])["Installs"]
    .sum()
    .reset_index()
    .sort_values(["Category", "Month"])
)

monthly_installs.head()

,Category,Month,Installs
0,BEAUTY,2018-03-01,5000.0
1,BEAUTY,2018-06-01,1000000.0
2,BEAUTY,2018-07-01,1000000.0
3,BEAUTY,2018-08-01,100000.0
4,BOOKS_AND_REFERENCE,2014-10-01,500000.0


### Note on time series methodology
The dataset has no historical install log — only a single `Last Updated` date per app. 
Monthly installs shown here are the sum of installs for apps last updated in that month, 
used as a documented proxy for an install trend over time (same estimation approach used 
in Task 2's choropleth). This is not a true install-growth time series.

In [40]:
monthly_installs["Growth_Pct"] = (
    monthly_installs.groupby("Category")["Installs"].pct_change() * 100
)

monthly_installs["Significant_Growth"] = monthly_installs["Growth_Pct"] > 20

monthly_installs.head(10)

,Category,Month,Installs,Growth_Pct,Significant_Growth
0,BEAUTY,2018-03-01,5000.0,NaN,False
1,BEAUTY,2018-06-01,1000000.0,19900.000000,True
2,BEAUTY,2018-07-01,1000000.0,0.000000,False
3,BEAUTY,2018-08-01,100000.0,-90.000000,False
4,BOOKS_AND_REFERENCE,2014-10-01,500000.0,NaN,False
5,BOOKS_AND_REFERENCE,2014-11-01,5000000.0,900.000000,True
6,BOOKS_AND_REFERENCE,2015-07-01,10000000.0,100.000000,True
7,BOOKS_AND_REFERENCE,2016-06-01,60000.0,-99.400000,False
8,BOOKS_AND_REFERENCE,2016-08-01,100000.0,66.666667,True
9,BOOKS_AND_REFERENCE,2017-02-01,100000.0,0.000000,False


In [41]:
translation_map_ts = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Verabredung"   # kept for completeness; DATING is excluded by the E/C/B filter,
                              # so this will never actually appear on this chart — flagged above.
}

monthly_installs["Category_Display"] = monthly_installs["Category"].replace(translation_map_ts)

In [42]:
india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour

if 18 <= current_hour < 21:

    fig3 = px.line(
        monthly_installs,
        x="Month",
        y="Installs",
        color="Category_Display",
        markers=True,
        title="Monthly Installs Trend by Category (Proxy via Last Updated Date)"
    )

    # Shade areas under curve where MoM growth > 20%, per category
    import plotly.graph_objects as go

    for cat in monthly_installs["Category_Display"].unique():
        cat_df = monthly_installs[monthly_installs["Category_Display"] == cat].sort_values("Month")
        sig = cat_df[cat_df["Significant_Growth"]]

        for _, row in sig.iterrows():
            month_idx = cat_df[cat_df["Month"] == row["Month"]].index[0]
            pos = cat_df.index.get_loc(month_idx)
            if pos == 0:
                continue
            prev_row = cat_df.iloc[pos - 1]

            fig3.add_trace(go.Scatter(
                x=[prev_row["Month"], row["Month"]],
                y=[prev_row["Installs"], row["Installs"]],
                fill="tozeroy",
                mode="none",
                fillcolor="rgba(255, 0, 0, 0.2)",
                showlegend=False,
                hoverinfo="skip"
            ))

    fig3.update_layout(
        xaxis_title="Month",
        yaxis_title="Total Installs",
        width=1100,
        height=600
    )

    fig3.show()

    os.makedirs("outputs", exist_ok=True)
    fig3.write_html("outputs/task3_timeseries.html")

else:
    print("This chart is only available between 6 PM and 9 PM IST.")

**Note on the 2018 spike:** The sharp spike near mid-2018 reflects a clustering artifact in 
the `Last Updated` field — most apps in this dataset were last updated shortly before the 
data was scraped, so installs "pile up" in that period rather than growing gradually. This 
is a known limitation of using `Last Updated` as a time-series proxy (see earlier note) and 
should not be read as real-world install acceleration.

**Insight:** Despite the clustering limitation, Entertainment and Communication show the 
most updates around 2018 — suggesting these categories saw the most active developer 
maintenance, plausible given how competitive and fast-moving both categories are.

In [43]:
task4_filtered = apps_df[
    (apps_df["Rating"] >= 4.2) &
    (~apps_df["App"].str.contains(r"\d", na=False)) &
    (apps_df["Category"].str.startswith(("T", "P"))) &
    (apps_df["Reviews"] > 1000) &
    (apps_df["Size_MB"].between(20, 80))
].copy()

task4_filtered.shape

(110, 17)

In [44]:
task4_filtered["Month"] = task4_filtered["Last Updated"].dt.to_period("M").dt.to_timestamp()

monthly_installs_t4 = (
    task4_filtered.groupby(["Category", "Month"])["Installs"]
    .sum()
    .reset_index()
    .sort_values(["Category", "Month"])
)

monthly_installs_t4["Cumulative_Installs"] = (
    monthly_installs_t4.groupby("Category")["Installs"].cumsum()
)

monthly_installs_t4.head()

,Category,Month,Installs,Cumulative_Installs
0,PARENTING,2018-03-01,100000.0,100000.0
1,PARENTING,2018-05-01,10000000.0,10100000.0
2,PARENTING,2018-07-01,600000.0,10700000.0
3,PERSONALIZATION,2016-12-01,1000000.0,1000000.0
4,PERSONALIZATION,2017-09-01,1000000.0,2000000.0


In [45]:
monthly_installs_t4["Growth_Pct"] = (
    monthly_installs_t4.groupby("Category")["Installs"].pct_change() * 100
)
monthly_installs_t4["Significant_Growth"] = monthly_installs_t4["Growth_Pct"] > 25

monthly_installs_t4.head(10)

,Category,Month,Installs,Cumulative_Installs,Growth_Pct,Significant_Growth
0,PARENTING,2018-03-01,100000.0,100000.0,NaN,False
1,PARENTING,2018-05-01,10000000.0,10100000.0,9900.000000,True
2,PARENTING,2018-07-01,600000.0,10700000.0,-94.000000,False
3,PERSONALIZATION,2016-12-01,1000000.0,1000000.0,NaN,False
4,PERSONALIZATION,2017-09-01,1000000.0,2000000.0,0.000000,False
5,PERSONALIZATION,2018-01-01,10000000.0,12000000.0,900.000000,True
6,PERSONALIZATION,2018-02-01,10000000.0,22000000.0,0.000000,False
7,PERSONALIZATION,2018-06-01,10000000.0,32000000.0,0.000000,False
8,PERSONALIZATION,2018-07-01,11500000.0,43500000.0,15.000000,False
9,PERSONALIZATION,2018-08-01,20000000.0,63500000.0,73.913043,True


### Notes on Task 4 methodology
- **Time proxy:** As in Task 3, `Last Updated` is used as a proxy for time since the 
  dataset has no true install-history log.
- **Growth basis:** The 25% MoM growth threshold is computed on *monthly* install totals, 
  not the cumulative curve — a cumulative series is monotonically non-decreasing, so 
  computing % change on it would not meaningfully capture "growth spikes."
- **Why Matplotlib instead of Plotly:** The task requires increasing color intensity 
  within a single category's area band for high-growth months. Plotly's `px.area()` 
  cannot vary fill opacity within one continuous band. Matplotlib's `fill_between()` 
  allows true per-segment opacity, so this chart is built in Matplotlib instead — the 
  only static (non-interactive) chart in this dashboard, used specifically to satisfy 
  this requirement literally rather than approximating it.

In [46]:
import matplotlib.font_manager as fm

for font_name in ['Yu Gothic', 'MS Gothic', 'Meiryo', 'Noto Sans CJK JP']:
    if any(font_name.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break

In [47]:
translation_map_t4 = {
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}

monthly_installs_t4["Category_Display"] = monthly_installs_t4["Category"].replace(translation_map_t4)

In [48]:
india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour
if 16 <= current_hour < 18:

    fig4, ax = plt.subplots(figsize=(12, 6))

    categories_t4 = monthly_installs_t4["Category_Display"].unique()
    colors = plt.cm.tab10.colors

    for i, cat in enumerate(categories_t4):
        cat_df = monthly_installs_t4[monthly_installs_t4["Category_Display"] == cat].sort_values("Month")
        base_color = colors[i % len(colors)]

        for j in range(1, len(cat_df)):
            x_seg = [cat_df.iloc[j-1]["Month"], cat_df.iloc[j]["Month"]]
            y_seg = [cat_df.iloc[j-1]["Cumulative_Installs"], cat_df.iloc[j]["Cumulative_Installs"]]
            is_growth = cat_df.iloc[j]["Significant_Growth"]
            alpha_val = 0.9 if is_growth else 0.3

            ax.fill_between(x_seg, 0, y_seg, color=base_color, alpha=alpha_val,
                             label=cat if j == 1 else "")

    ax.set_title("Cumulative Installs Over Time by Category (T/P Categories, Proxy via Last Updated)")
    ax.set_xlabel("Month")
    ax.set_ylabel("Cumulative Installs")
    ax.legend(loc="upper left", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.xticks(rotation=45)
    plt.tight_layout()

    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/task4_stacked_area.png", dpi=150)
    plt.show()

else:
    print("This chart is only available between 4 PM and 6 PM IST.")

This chart is only available between 4 PM and 6 PM IST.


**Note:** As in Task 3, the sharp rise near mid-2018 reflects `Last Updated` clustering 
in the dataset, not real-world cumulative install acceleration.

**Note:** Extremely high growth percentages (e.g., 1000%+) occur when a category's 
prior-month installs are near zero, making percentage-based growth mathematically 
volatile at low baselines — expected behavior of `pct_change()`, not a data error.

**Note on visual overlap:** Since segments are drawn individually to allow per-segment 
opacity (see methodology above), category bands overlap rather than stack cleanly like a 
traditional stacked area chart. Categories with smaller totals (Parenting, Personalization) 
may be visually dominated by larger ones (Photography) as a result — a tradeoff of the 
opacity-highlighting approach versus a standard stacked area.

**Insight:** Photography shows the most concentrated growth-highlighted (high-opacity) 
segments, particularly in the final months near 2018 — suggesting installs for this 
category were driven by a small number of sharp updates-clusters rather than steady 
growth across the timeline.

In [49]:
task5_filtered = apps_df[
    (apps_df["Size_MB"] >= 10) &
    (apps_df["Last Updated"].dt.month == 1)
].copy()

task5_filtered.shape

(168, 17)

In [50]:
category_stats = (
    task5_filtered.groupby("Category")
    .agg(
        Avg_Rating=("Rating", "mean"),
        Total_Reviews=("Reviews", "sum"),
        Total_Installs=("Installs", "sum")
    )
    .reset_index()
)

category_stats.shape

(29, 4)

In [51]:
category_stats = category_stats[category_stats["Avg_Rating"] >= 4.0]
category_stats.shape

(19, 4)

In [52]:
top10_categories_t5 = category_stats.sort_values("Total_Installs", ascending=False).head(10)
top10_categories_t5

,Category,Avg_Rating,Total_Reviews,Total_Installs
25,SPORTS,4.342857,1982017,120511000.0
12,GAME,4.251515,2412245,117291000.0
9,FAMILY,4.188889,3006888,111220820.0
7,ENTERTAINMENT,4.300000,869111,21000000.0
20,PERSONALIZATION,4.475000,155996,15060000.0
15,LIFESTYLE,4.050000,53376,6171500.0
6,EDUCATION,4.400000,57645,2000000.0
4,COMMUNICATION,4.050000,15394,1010000.0
23,SHOPPING,4.200000,9975,1000000.0
0,ART_AND_DESIGN,4.100000,2167,620000.0


### Notes on Task 5 methodology
- **Filter order:** `Avg_Rating`, `Total_Reviews`, and `Total_Installs` are computed only 
  on apps that pass the size (≥10MB) and Last-Updated-in-January filters first. Average 
  rating and top-10 ranking therefore reflect this filtered subset, not the full dataset. 
  This interpretation was chosen for internal consistency and should be confirmed with 
  the mentor if a different order was intended.
- **"January" filter:** Interpreted as `Last Updated` month == January, across any year 
  (not restricted to a specific year), since the task didn't specify one.

In [ ]:
india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour

if 15 <= current_hour < 17:

    fig5 = make_subplots(specs=[[{"secondary_y": True}]])

    fig5.add_trace(
        go.Bar(
            x=top10_categories_t5["Category"],
            y=top10_categories_t5["Avg_Rating"],
            name="Average Rating",
            marker_color="steelblue",
            offsetgroup=0
        ),
        secondary_y=False
    )

    fig5.add_trace(
        go.Bar(
            x=top10_categories_t5["Category"],
            y=top10_categories_t5["Total_Reviews"],
            name="Total Reviews",
            marker_color="orange",
            offsetgroup=1
        ),
        secondary_y=True
    )

    fig5.update_layout(
        title="Top 10 Categories by Installs — Avg Rating vs Total Reviews (Size≥10MB, Jan Updates)",
        barmode="group",
        bargap=0.25,
        bargroupgap=0.15,
        width=1100,
        height=600,
        legend=dict(x=0.01, y=1.1, orientation="h")
    )

    fig5.update_yaxes(title_text="Average Rating", range=[0, 5], secondary_y=False)
    fig5.update_yaxes(title_text="Total Reviews", secondary_y=True, showgrid=False)
    fig5.update_xaxes(title_text="Category")

    fig5.show()

    os.makedirs("outputs", exist_ok=True)
    fig5.write_html("outputs/task5_grouped_bar.html")

else:
    print("This chart is only available between 3 PM and 5 PM IST.")

This chart is only available between 3 PM and 5 PM IST.


In [54]:
task6_filtered = apps_df[
    (apps_df["Installs"] >= 10000) &
    (apps_df["Android_Ver_Num"] > 4.0) &
    (apps_df["Size_MB"] > 15) &
    (apps_df["Content Rating"] == "Everyone") &
    (apps_df["App"].str.len() <= 30) &
    (
        ((apps_df["Type"] == "Paid") & (apps_df["Revenue"] >= 10000)) |
        (apps_df["Type"] == "Free")
    )
].copy()

task6_filtered.shape

(870, 17)

### Notes on Task 6 methodology
- **Revenue is a proxy** (`Price × Installs`) — the dataset has no true revenue field.
- **Revenue filter applies to Paid apps only.** Free apps have Price=0, so Revenue is 
  always 0 — applying the $10,000 revenue floor to all apps would eliminate every free 
  app and break the free-vs-paid comparison the task requires. Confirmed acceptable 
  with mentor.
- **Average Revenue for Free apps will show as $0** in the chart — this is expected, 
  not a data error, given the proxy formula above.
- **Top 3 categories** are determined by total installs within this filtered subset 
  (consistent with the approach used in Task 5).

In [55]:
top3_categories_t6 = (
    task6_filtered.groupby("Category")["Installs"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)
top3_categories_t6

['GAME', 'FAMILY', 'TOOLS']

In [56]:
task6_top3 = task6_filtered[task6_filtered["Category"].isin(top3_categories_t6)]

agg_t6 = (
    task6_top3.groupby(["Category", "Type"])
    .agg(Avg_Installs=("Installs", "mean"), Avg_Revenue=("Revenue", "mean"))
    .reset_index()
)

agg_t6["Cat_Type"] = agg_t6["Category"] + " (" + agg_t6["Type"] + ")"
agg_t6

,Category,Type,Avg_Installs,Avg_Revenue,Cat_Type
0,FAMILY,Free,6.456321e+06,0.000000,FAMILY (Free)
1,FAMILY,Paid,3.616667e+05,597216.666667,FAMILY (Paid)
2,GAME,Free,3.319490e+07,0.000000,GAME (Free)
3,GAME,Paid,6.714286e+04,276471.428571,GAME (Paid)
4,TOOLS,Free,2.219867e+07,0.000000,TOOLS (Free)
5,TOOLS,Paid,1.000000e+05,449000.000000,TOOLS (Paid)


In [ ]:
india_time = datetime.now(pytz.timezone("Asia/Kolkata"))
current_hour = india_time.hour

if 13 <= current_hour < 14:

    fig6 = make_subplots(specs=[[{"secondary_y": True}]])

    fig6.add_trace(
        go.Bar(
            x=agg_t6["Cat_Type"],
            y=agg_t6["Avg_Installs"],
            name="Avg Installs",
            marker_color="steelblue",
            offsetgroup=0
        ),
        secondary_y=False
    )

    fig6.add_trace(
        go.Bar(
            x=agg_t6["Cat_Type"],
            y=agg_t6["Avg_Revenue"],
            name="Avg Revenue ($)",
            marker_color="green",
            offsetgroup=1
        ),
        secondary_y=True
    )

    fig6.update_layout(
        title="Avg Installs vs Avg Revenue — Free vs Paid (Top 3 Categories)",
        barmode="group",
        bargap=0.25,
        bargroupgap=0.15,
        width=1100,
        height=600,
        legend=dict(x=0.01, y=1.1, orientation="h"),
        xaxis_tickangle=-30
    )

    fig6.update_yaxes(title_text="Average Installs", secondary_y=False)
    fig6.update_yaxes(title_text="Average Revenue ($)", secondary_y=True, showgrid=False)
    fig6.update_xaxes(title_text="Category (Type)")

    fig6.show()

    os.makedirs("outputs", exist_ok=True)
    fig6.write_html("outputs/task6_dual_axis.html")

else:
    print("This chart is only available between 1 PM and 2 PM IST.")

This chart is only available between 1 PM and 2 PM IST.
